# Coral Reef AI — Classification Notebook

**Model:** MobileNetV4 Conv Small 0.5  
**Goal:** Build the 2,198 ROI classification dataset, validate leakage-safe splits, train a lightweight 6-class classifier, and evaluate on Validation only.

Expected split:
- Train: 1,540 ROIs
- Validation: 354 ROIs
- Test: 304 ROIs
- Total: 2,198 ROIs

> If this runs from a second Google account, make sure `Coral_Reef_AI_Project` is shared with that account and accessible in My Drive. If the path differs, edit `PROJECT_ROOT` in Cell 2 only.


In [ ]:
# CELL 1 — INSTALL
!pip -q install -U timm scikit-learn


In [ ]:
# CELL 2 — MOUNT DRIVE + ENVIRONMENT + PATHS
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path
from PIL import Image
import math, json, random, shutil, time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from torchvision import datasets, transforms
import timm
from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix

SEED=42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
if torch.cuda.is_available(): torch.cuda.manual_seed_all(SEED)

if not torch.cuda.is_available():
    raise RuntimeError('Enable GPU in Colab: Runtime -> Change runtime type -> GPU')

DEVICE=torch.device('cuda')
GPU_NAME=torch.cuda.get_device_name(0)
VRAM=torch.cuda.get_device_properties(0).total_memory/1024**3

PROJECT_ROOT=Path('/content/drive/MyDrive/Coral_Reef_AI_Project')
MANIFEST=PROJECT_ROOT/'04_Dataset_Splits'/'classification_roi_manifest_split.csv'
DET_IMAGES=PROJECT_ROOT/'01_Data'/'Final'/'Detection_v1'/'images'
LOCAL_ROOT=Path('/content/Coral_Classification_v1')
META_ROOT=PROJECT_ROOT/'01_Data'/'Final'/'Classification_v1'/'metadata'
EXP_ROOT=PROJECT_ROOT/'06_Experiments'/'MobileNetV4_ConvSmall050_Classification_15Epochs'
META_ROOT.mkdir(parents=True, exist_ok=True)
EXP_ROOT.mkdir(parents=True, exist_ok=True)

print('GPU:',GPU_NAME, f'| VRAM={VRAM:.2f} GB')
print('Project:',PROJECT_ROOT)
print('Manifest:',MANIFEST)

if not MANIFEST.exists():
    raise FileNotFoundError(f'Manifest not found: {MANIFEST}')
if not DET_IMAGES.exists():
    raise FileNotFoundError(f'Detection_v1 images not found: {DET_IMAGES}')

df=pd.read_csv(MANIFEST)
req={'roi_id','image_stem','raw_detection_label','x1_px','y1_px','x2_px','y2_px','split'}
missing=req-set(df.columns)
if missing: raise RuntimeError(f'Missing manifest columns: {sorted(missing)}')

assert len(df)==2198
assert df['roi_id'].nunique()==2198
assert df['image_stem'].nunique()==646
assert df['raw_detection_label'].nunique()==6
assert set(df['split'].unique())=={'train','validation','test'}
print('STATUS: INPUT_MANIFEST_OK')


In [ ]:
# CELL 3 — BUILD + VALIDATE CLASSIFICATION CROPS
if LOCAL_ROOT.exists(): shutil.rmtree(LOCAL_ROOT)

SPLIT_MAP={'train':'train','validation':'val','test':'test'}
CLASSES=sorted(df['raw_detection_label'].astype(str).unique().tolist())
for sp in ['train','val','test']:
    for c in CLASSES:
        (LOCAL_ROOT/sp/c).mkdir(parents=True, exist_ok=True)

records=[]; errors=[]
for (split_name,image_stem), g in df.groupby(['split','image_stem']):
    src=DET_IMAGES/split_name/f'{image_stem}.jpg'
    if not src.exists():
        errors.append((image_stem,'missing source')); continue
    try:
        with Image.open(src) as im:
            img=im.convert('RGB')
        W,H=img.size
        for _,r in g.iterrows():
            x1=max(0,int(math.floor(float(r.x1_px))))
            y1=max(0,int(math.floor(float(r.y1_px))))
            x2=min(W,int(math.ceil(float(r.x2_px))))
            y2=min(H,int(math.ceil(float(r.y2_px))))
            if x2<=x1 or y2<=y1:
                errors.append((r.roi_id,'bad bbox')); continue
            crop=img.crop((x1,y1,x2,y2))
            cls=str(r.raw_detection_label)
            out=LOCAL_ROOT/SPLIT_MAP[split_name]/cls/f'{r.roi_id}.jpg'
            crop.save(out,'JPEG',quality=95,subsampling=0)
            records.append({
                'roi_id':r.roi_id,'image_stem':image_stem,'class':cls,
                'split':SPLIT_MAP[split_name],'crop_path':str(out),
                'crop_width':crop.width,'crop_height':crop.height
            })
    except Exception as e:
        errors.append((image_stem,repr(e)))

if errors:
    print(errors[:20]); raise RuntimeError(f'Crop build errors: {len(errors)}')

crop_df=pd.DataFrame(records)
assert len(crop_df)==2198

expected={
'Euphflfiaancora':{'train':298,'val':62,'test':48},
'Favosites':{'train':165,'val':24,'test':32},
'Platygyra':{'train':130,'val':30,'test':25},
'Sarcophyton':{'train':232,'val':64,'test':43},
'Sinularia':{'train':510,'val':136,'test':114},
'WavingHand':{'train':205,'val':38,'test':42},
}

ct=pd.crosstab(crop_df['class'],crop_df['split'])
for c,parts in expected.items():
    for sp,n in parts.items():
        assert int(ct.loc[c,sp])==n,(c,sp,ct.loc[c,sp],n)

assert int((crop_df['split']=='train').sum())==1540
assert int((crop_df['split']=='val').sum())==354
assert int((crop_df['split']=='test').sum())==304

tr=set(crop_df.loc[crop_df['split']=='train','image_stem'])
va=set(crop_df.loc[crop_df['split']=='val','image_stem'])
te=set(crop_df.loc[crop_df['split']=='test','image_stem'])
assert not (tr&va) and not (tr&te) and not (va&te)

CROP_MANIFEST=META_ROOT/'classification_crop_manifest.csv'
crop_df.to_csv(CROP_MANIFEST,index=False)
pd.DataFrame({'class_id':range(6),'operational_class_name':CLASSES}).to_csv(META_ROOT/'classification_class_map.csv',index=False)

print(ct)
print('Total:',len(crop_df))
print('Leakage overlaps:',len(tr&va),len(tr&te),len(va&te))
print('STATUS: CLASSIFICATION_DATASET_READY')


In [ ]:
# CELL 4 — MODEL + TRANSFORMS + DATALOADERS
MODEL_NAME='mobilenetv4_conv_small_050.e3000_r224_in1k'
NUM_CLASSES=6
INPUT_SIZE=224

model=timm.create_model(MODEL_NAME,pretrained=True,num_classes=NUM_CLASSES).to(DEVICE)
params=sum(p.numel() for p in model.parameters())
print('Model:',MODEL_NAME)
print(f'Params: {params/1e6:.3f} M')

data_cfg=timm.data.resolve_model_data_config(model)
MEAN=data_cfg['mean']; STD=data_cfg['std']

class MildUnderwaterColorTensor:
    def __init__(self,p=0.25): self.p=p
    def __call__(self,t):
        if random.random()>=self.p: return t
        out=t.clone()
        out[0]*=random.uniform(0.78,0.97)
        out[1]*=random.uniform(0.95,1.10)
        out[2]*=random.uniform(0.98,1.13)
        return out.clamp(0,1)

train_tf=transforms.Compose([
    transforms.RandomResizedCrop(INPUT_SIZE,scale=(0.75,1.0),ratio=(0.85,1.15)),
    transforms.RandomHorizontalFlip(0.5),
    transforms.RandomVerticalFlip(0.15),
    transforms.RandomRotation(10),
    transforms.ColorJitter(brightness=0.22,contrast=0.22,saturation=0.20,hue=0.04),
    transforms.RandomApply([transforms.GaussianBlur(3,sigma=(0.1,1.5))],p=0.12),
    transforms.ToTensor(),
    MildUnderwaterColorTensor(0.25),
    transforms.Normalize(MEAN,STD),
])
val_tf=timm.data.create_transform(**data_cfg,is_training=False)

train_ds=datasets.ImageFolder(LOCAL_ROOT/'train',transform=train_tf)
val_ds=datasets.ImageFolder(LOCAL_ROOT/'val',transform=val_tf)
test_ds=datasets.ImageFolder(LOCAL_ROOT/'test',transform=val_tf)
assert list(train_ds.class_to_idx.keys())==CLASSES
assert len(train_ds)==1540 and len(val_ds)==354 and len(test_ds)==304

counts=np.bincount(np.array(train_ds.targets),minlength=6).astype(float)
weights=1/np.sqrt(counts); weights=weights/weights.mean()
class_weights=torch.tensor(weights,dtype=torch.float32,device=DEVICE)

BATCH=64 if VRAM>=12 else 32
train_loader=DataLoader(train_ds,batch_size=BATCH,shuffle=True,num_workers=2,pin_memory=True,persistent_workers=True)
val_loader=DataLoader(val_ds,batch_size=BATCH,shuffle=False,num_workers=2,pin_memory=True,persistent_workers=True)
test_loader=DataLoader(test_ds,batch_size=BATCH,shuffle=False,num_workers=2,pin_memory=True,persistent_workers=True)

print('class_to_idx:',train_ds.class_to_idx)
for i,c in enumerate(CLASSES): print(f'{c:<20} count={int(counts[i]):<4} weight={weights[i]:.3f}')
print('Batch:',BATCH)
print('STATUS: CLASSIFICATION_LOADERS_READY')


In [ ]:
# CELL 5 — TRAIN: 3 EPOCH HEAD + 12 EPOCH FULL FINE-TUNE
LOSS=nn.CrossEntropyLoss(weight=class_weights,label_smoothing=0.05)
BEST=EXP_ROOT/'best_model_macro_f1.pth'
HIST=EXP_ROOT/'training_history.csv'
AMP=True

def run_epoch(model,loader,optimizer=None,scaler=None):
    train=optimizer is not None
    model.train(train)
    losses=[]; ys=[]; ps=[]
    for x,y in loader:
        x=x.to(DEVICE,non_blocking=True); y=y.to(DEVICE,non_blocking=True)
        if train: optimizer.zero_grad(set_to_none=True)
        with torch.autocast(device_type='cuda',dtype=torch.float16,enabled=AMP):
            logits=model(x); loss=LOSS(logits,y)
        if train:
            scaler.scale(loss).backward(); scaler.step(optimizer); scaler.update()
        pred=logits.argmax(1)
        losses.append(float(loss.detach().cpu()))
        ys.extend(y.detach().cpu().tolist()); ps.extend(pred.detach().cpu().tolist())
    return {
        'loss':float(np.mean(losses)),
        'accuracy':float(accuracy_score(ys,ps)),
        'macro_f1':float(f1_score(ys,ps,average='macro',zero_division=0)),
        'weighted_f1':float(f1_score(ys,ps,average='weighted',zero_division=0)),
    }

history=[]; best_f1=-1; best_epoch=-1; epoch=0; start=time.time()

def maybe_save(stage,valm,opt):
    global best_f1,best_epoch
    if valm['macro_f1']>best_f1:
        best_f1=valm['macro_f1']; best_epoch=epoch
        torch.save({
            'model_name':MODEL_NAME,'model_state_dict':model.state_dict(),
            'class_to_idx':train_ds.class_to_idx,'epoch':epoch,'stage':stage,
            'val_macro_f1':best_f1,'input_size':INPUT_SIZE,'mean':MEAN,'std':STD
        },BEST)
        print(f'  NEW BEST Macro-F1={best_f1:.4f}')

# Stage 1
for p in model.parameters(): p.requires_grad=False
for p in model.get_classifier().parameters(): p.requires_grad=True
opt=torch.optim.AdamW([p for p in model.parameters() if p.requires_grad],lr=1e-3,weight_decay=1e-4)
scaler=torch.amp.GradScaler('cuda',enabled=AMP)
print('STAGE 1: head-only, 3 epochs')
for _ in range(3):
    epoch+=1
    tm=run_epoch(model,train_loader,opt,scaler)
    with torch.no_grad(): vm=run_epoch(model,val_loader)
    history.append({'epoch':epoch,'stage':'head_only','lr':opt.param_groups[0]['lr'],**{f'train_{k}':v for k,v in tm.items()},**{f'val_{k}':v for k,v in vm.items()}})
    print(f'Epoch {epoch:02d}/15 | val_acc={vm["accuracy"]:.4f} | val_macroF1={vm["macro_f1"]:.4f}')
    maybe_save('head_only',vm,opt)

# Stage 2
for p in model.parameters(): p.requires_grad=True
opt=torch.optim.AdamW(model.parameters(),lr=2e-4,weight_decay=1e-4)
sched=torch.optim.lr_scheduler.CosineAnnealingLR(opt,T_max=12,eta_min=1e-6)
scaler=torch.amp.GradScaler('cuda',enabled=AMP)
print('STAGE 2: full fine-tune, 12 epochs')
for _ in range(12):
    epoch+=1
    tm=run_epoch(model,train_loader,opt,scaler)
    with torch.no_grad(): vm=run_epoch(model,val_loader)
    history.append({'epoch':epoch,'stage':'full_finetune','lr':opt.param_groups[0]['lr'],**{f'train_{k}':v for k,v in tm.items()},**{f'val_{k}':v for k,v in vm.items()}})
    print(f'Epoch {epoch:02d}/15 | val_acc={vm["accuracy"]:.4f} | val_macroF1={vm["macro_f1"]:.4f}')
    maybe_save('full_finetune',vm,opt)
    sched.step()

pd.DataFrame(history).to_csv(HIST,index=False)
print(f'Training time: {(time.time()-start)/60:.2f} min')
print('Best epoch:',best_epoch,'| Best Macro-F1:',round(best_f1,4))
print('Best model:',BEST)
print('STATUS: CLASSIFICATION_TRAINING_COMPLETE')


In [ ]:
# CELL 6 — BEST MODEL VALIDATION + CONFUSION MATRIX
ckpt=torch.load(BEST,map_location=DEVICE,weights_only=False)
best_model=timm.create_model(MODEL_NAME,pretrained=False,num_classes=NUM_CLASSES).to(DEVICE)
best_model.load_state_dict(ckpt['model_state_dict']); best_model.eval()

ys=[]; ps=[]
with torch.no_grad():
    for x,y in val_loader:
        pred=best_model(x.to(DEVICE)).argmax(1).cpu()
        ys.extend(y.tolist()); ps.extend(pred.tolist())

acc=accuracy_score(ys,ps)
macro=f1_score(ys,ps,average='macro',zero_division=0)
weighted=f1_score(ys,ps,average='weighted',zero_division=0)
report=classification_report(ys,ps,target_names=CLASSES,zero_division=0)
print('Accuracy   :',round(acc,4))
print('Macro-F1   :',round(macro,4))
print('Weighted-F1:',round(weighted,4))
print('
',report)

metrics={'model':MODEL_NAME,'best_epoch':int(ckpt['epoch']),'validation_accuracy':float(acc),'validation_macro_f1':float(macro),'validation_weighted_f1':float(weighted),'test_evaluated':False}
with open(EXP_ROOT/'validation_metrics.json','w',encoding='utf-8') as f: json.dump(metrics,f,indent=2)

cm=confusion_matrix(ys,ps,labels=list(range(6)))
fig,ax=plt.subplots(figsize=(10,8)); ax.imshow(cm)
ax.set_xticks(range(6)); ax.set_yticks(range(6)); ax.set_xticklabels(CLASSES,rotation=45,ha='right'); ax.set_yticklabels(CLASSES)
ax.set_xlabel('Predicted'); ax.set_ylabel('True'); ax.set_title('Validation Confusion Matrix')
for i in range(6):
    for j in range(6): ax.text(j,i,str(cm[i,j]),ha='center',va='center')
fig.tight_layout(); fig.savefig(EXP_ROOT/'validation_confusion_matrix.png',dpi=180,bbox_inches='tight'); plt.show()

print('Best checkpoint:',BEST)
print('TEST SET WAS NOT EVALUATED.')
print('STATUS: CLASSIFICATION_BASELINE_COMPLETE')


In [ ]:
# CELL 7 — TRAINING CURVES + FINAL SUMMARY
h=pd.read_csv(HIST)
fig,ax=plt.subplots(figsize=(10,6))
ax.plot(h.epoch,h.train_loss,marker='o',label='Train loss')
ax.plot(h.epoch,h.val_loss,marker='o',label='Val loss')
ax.set_xlabel('Epoch'); ax.set_ylabel('Loss'); ax.set_title('Classification Loss'); ax.legend(); ax.grid(True,alpha=.3)
fig.tight_layout(); fig.savefig(EXP_ROOT/'loss_curve.png',dpi=180,bbox_inches='tight'); plt.show()

fig,ax=plt.subplots(figsize=(10,6))
ax.plot(h.epoch,h.val_accuracy,marker='o',label='Val accuracy')
ax.plot(h.epoch,h.val_macro_f1,marker='o',label='Val Macro-F1')
ax.set_xlabel('Epoch'); ax.set_ylabel('Score'); ax.set_title('Validation Metrics'); ax.legend(); ax.grid(True,alpha=.3)
fig.tight_layout(); fig.savefig(EXP_ROOT/'validation_metrics_curve.png',dpi=180,bbox_inches='tight'); plt.show()

print('='*100)
print('FINAL SUMMARY')
print('='*100)
print('Model:',MODEL_NAME)
print(f'Parameters: {params/1e6:.3f} M')
print('Train/Val/Test:',len(train_ds),len(val_ds),len(test_ds))
print('Best epoch:',ckpt['epoch'])
print('Val Accuracy:',round(acc,4))
print('Val Macro-F1:',round(macro,4))
print('Best checkpoint:',BEST)
print('Experiment:',EXP_ROOT)
print('Internal TEST set remains untouched.')
